# 项目二：墨读 App 转化漏斗与用户留存分析

数据是固定随机种子生成的**虚构教学数据**（`data/app/`），不代表任何真实 App 业绩。你将完成全量漏斗、分渠道/分设备定位、留存率、周同期群热力图以及 6 道独立自动检查。

请选择 `Python (da-course)` 内核，从本包 `notebooks/` 目录打开。

In [ ]:
import sys
sys.path.insert(0, "..")
from dacheck import check, hint, setup_chinese_font
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 1. 读取数据与质量体检

In [ ]:
users = pd.read_csv("../data/app/app_users.csv", parse_dates=["register_date"], encoding="utf-8-sig")
events = pd.read_csv("../data/app/app_events.csv", parse_dates=["event_time"], encoding="utf-8-sig")
active = pd.read_csv("../data/app/app_active.csv", parse_dates=["active_date"], encoding="utf-8-sig")

assert users["user_id"].is_unique
assert set(events["user_id"]).issubset(set(users["user_id"]))
assert set(active["user_id"]).issubset(set(users["user_id"]))
assert (active["active_date"].max() - users["register_date"].max()).days >= 30
print("质量体检通过：", {"users": users.shape, "events": events.shape, "active": active.shape})

## 2. 全量五步转化漏斗

练习 P-2.1：按 `["app_open", "view_course", "start_trial", "click_pay", "pay_success"]` 顺序计算 `step, users, step_rate, overall_rate`。

In [ ]:
p2_funnel = ...
check("P-2.1", p2_funnel)

独立可视化全量漏斗。

In [ ]:
setup_chinese_font()
steps_ref = ["app_open", "view_course", "start_trial", "click_pay", "pay_success"]
cnt_ref = [events.loc[events["event_name"].eq(st), "user_id"].nunique() for st in steps_ref]
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(steps_ref[::-1], cnt_ref[::-1], color="#0ea5e9")
ax.set(title="墨读 App 全量五步转化漏斗（虚构数据）", xlabel="唯一用户数", ylabel="")
for i, v in enumerate(cnt_ref[::-1]):
    ax.text(v + 80, i, f"{v} ({v / cnt_ref[0]:.1%})", va="center")
ax.set_xlim(0, 7200)
fig.tight_layout()
plt.show()

## 3. 分渠道与分设备下钻定位瓶颈

练习 P-2.2：按 `channel` 统计 `open_users, trial_users, click_users, pay_users, trial_to_click, overall_pay_rate`，按 `overall_pay_rate` 降序排列。

In [ ]:
p2_channel = ...
check("P-2.2", p2_channel)

练习 P-2.3：按 `device` 统计 `click_users, pay_users, pay_rate = pay_users / click_users`，按 `device` 升序排列。

In [ ]:
p2_device = ...
check("P-2.3", p2_device)

## 4. 分渠道留存与按周同期群

练习 P-2.4：按 `channel` 统计 `users, d1, d7, d30`（留存率小数），按 `d7` 降序排列。

In [ ]:
p2_ch_ret = ...
check("P-2.4", p2_ch_ret)

练习 P-2.5：将 `register_date` 对齐到当周周一 `cohort_week`（`YYYY-MM-DD`），统计每周同期群的 `users` 和 `d7_rate`，按 `cohort_week` 升序排列。

In [ ]:
p2_weekly = ...
check("P-2.5", p2_weekly)

绘制周同期群留存热力图（独立示例，看 D1/D3/D7/D14/D30）。

In [ ]:
u_hm = users.copy()
u_hm["cohort_week"] = (u_hm["register_date"] - pd.to_timedelta(u_hm["register_date"].dt.day_of_week, unit="D")).dt.strftime("%Y-%m-%d")
au_hm = active.merge(u_hm[["user_id", "register_date", "cohort_week"]], on="user_id", how="left")
au_hm["day_diff"] = (au_hm["active_date"] - au_hm["register_date"]).dt.days
base_hm = u_hm.groupby("cohort_week")["user_id"].nunique()
mat = pd.DataFrame({
    f"D{d}": au_hm.loc[au_hm["day_diff"].eq(d)].groupby("cohort_week")["user_id"].nunique() / base_hm
    for d in [1, 3, 7, 14, 30]
}).fillna(0)
fig, ax = plt.subplots(figsize=(7.5, 5))
sns.heatmap(mat * 100, annot=True, fmt=".1f", cmap="YlGnBu", ax=ax, cbar_kws={"label": "留存率 (%)"})
ax.set(title="墨读 App 按周同期群留存热力图（虚构数据）", xlabel="留存窗口", ylabel="注册周（周一）")
fig.tight_layout()
plt.show()

## 5. 项目二核心指标卡

练习 P-2.6：汇总 6 键指标卡字典：`total_users, overall_pay_rate, best_channel, worst_channel, ios_pay_step_rate, android_pay_step_rate`。

In [ ]:
p2_kpi = ...
check("P-2.6", p2_kpi)